In [4]:
import pandas as pd
import numpy as np
import sys

import json
with open('config.json', 'r') as f:
    config = json.load(f)

sys.path.insert(0, config['utils_path'])

from sklearn.pipeline import Pipeline
from sklearn.svm import SVR, NuSVR  # regression
from sklearn.neighbors import KNeighborsRegressor

import warnings
warnings.filterwarnings('ignore')

from helpers.modeling import (
    prepare_data,
    identify_column_types,
    create_preprocessor,
    evaluate_model,
    run_grid_search,
)

# UHPC Modelling & Hyperparameter Tuning

## 1. Imports

In [5]:
raw = pd.read_csv(config["initial_cleaned_filepath"])
df_raw_20 = pd.read_csv(config["raw_dropped_features_20_filepath"])
df_raw_50 = pd.read_csv(config["raw_dropped_features_50_filepath"])
df_recoded_20 = pd.read_csv(config["semantic_recoding_features_20_filepath"])
df_recoded_50 = pd.read_csv(config["semantic_recoding_features_50_filepath"])

df_recoded_50 = df_recoded_50.drop(columns=['cement_grade']) # cement grade 38% missing, encodes sames info as cement type
fiber_cols = ['fiber1_length', 'fiber1_diameter']  
df_recoded_50[fiber_cols] = df_recoded_50[fiber_cols].fillna(0)  # replacing with mean wrong

dfs = [raw,df_raw_20, df_raw_50, df_recoded_20, df_recoded_50]
df_names = ['raw','raw_20', 'raw_50', 'recoded_20', 'recoded_50']
models = [KNeighborsRegressor, NuSVR, SVR]
model_names_list = ['KNeighborsRegressor', 'NuSVR', 'SVR']

## 2. Load & Prepare Data

## 3. Utility Functions

**Key Pipeline Function (from `helpers/modeling.py`):**
- `create_preprocessor()` creates ColumnTransformer with: numerical scaling + OneHotEncoder + TargetEncoder

In [6]:
for df, df_name in zip(dfs, df_names):
    for model_class, model_name in zip(models, model_names_list):
        
        target_col = 'cs_28d'
        X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(df, target_col)
        
        X = df.drop(columns=[target_col])
        numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)
        
        preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns)
        
        full_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model_class())
        ])
        
        full_pipeline.fit(X_train, y_train)
        
        y_pred_train = full_pipeline.predict(X_train)
        y_pred_test = full_pipeline.predict(X_test)
        
        train_metrics = evaluate_model(y_train, y_pred_train)
        test_metrics = evaluate_model(y_test, y_pred_test)
        
        rmse_train = train_metrics['RMSE']
        rmse_test = test_metrics['RMSE']
        mae = test_metrics['MAE']
        r2 = test_metrics['R2']
        r = test_metrics['Correlation']
        
        print(f"Dataset: {df_name:12} | Model: {model_name:20} | RMSE test: {rmse_test:.4f} | RMSE train: {rmse_train:.4f} | MAE: {mae:.4f} | R2: {r2:.4f} | R: {r:.4f}")

Dataset: raw          | Model: KNeighborsRegressor  | RMSE test: 17.5666 | RMSE train: 14.8660 | MAE: 13.1533 | R2: 0.7362 | R: 0.8595
Dataset: raw          | Model: NuSVR                | RMSE test: 27.3957 | RMSE train: 28.1088 | MAE: 20.8514 | R2: 0.3583 | R: 0.6926
Dataset: raw          | Model: SVR                  | RMSE test: 26.5396 | RMSE train: 27.0007 | MAE: 19.8706 | R2: 0.3978 | R: 0.6954
Dataset: raw_20       | Model: KNeighborsRegressor  | RMSE test: 19.3998 | RMSE train: 16.7973 | MAE: 14.5735 | R2: 0.6782 | R: 0.8250
Dataset: raw_20       | Model: NuSVR                | RMSE test: 28.1460 | RMSE train: 28.8791 | MAE: 21.7737 | R2: 0.3227 | R: 0.6320
Dataset: raw_20       | Model: SVR                  | RMSE test: 27.5464 | RMSE train: 28.1356 | MAE: 21.0087 | R2: 0.3512 | R: 0.6391
Dataset: raw_50       | Model: KNeighborsRegressor  | RMSE test: 17.4036 | RMSE train: 13.9743 | MAE: 13.0618 | R2: 0.7410 | R: 0.8621
Dataset: raw_50       | Model: NuSVR                | R

## 4. Model Comparison Loop

**Pipeline Created:**
```python
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),  # Uses create_preprocessor()
    ('model', model_class())
])
```

## 5. Grid Search Helper Function

**Purpose:** Automates hyperparameter tuning for any pipeline (`run_grid_search` from `helpers/modeling.py`)

In [7]:
df = df_recoded_50
target_col = 'cs_28d'

X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(df, target_col)

print(f"Train set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

X = df.drop(columns=[target_col])
numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)

preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns,
                                   handle_unknown='ignore')

Train set size: 1451
Validation set size: 311
Test set size: 311


## 6. Hyperparameter Tuning Setup

**Preprocessing Pipeline Created:**
```python
preprocessor = create_preprocessor(...)  # ColumnTransformer
```
Used for all 3 models below (KNN, SVR, NuSVR)

### 6.1 KNN - Pipeline & Tuning

In [8]:
# KNN PIPELINE CREATION
knn_pipeline = Pipeline([('preprocessor', preprocessor), ('model', KNeighborsRegressor())])

knn_grid = {
    'model__n_neighbors': list(range(1, 31)),
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2, 3],
    'model__metric': ['euclidean', 'manhattan', 'minkowski']
}

gs_knn = run_grid_search(knn_pipeline, knn_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'KNN')


GridSearchCV will test 540 combinations for KNN...

BEST HYPERPARAMETERS — KNN
  metric: minkowski
  n_neighbors: 3
  p: 1
  weights: distance
Best CV RMSE: 15.9142

VALIDATION SET PERFORMANCE
RMSE: 15.9252
MAE: 11.3549
R2: 0.8218
Correlation: 0.9105

TEST SET PERFORMANCE
RMSE: 16.0832
MAE: 11.1342
R2: 0.7788
Correlation: 0.8838

RESULTS SUMMARY
          Metric  Validation      Test
           RMSE   15.925163 16.083224
            MAE   11.354857 11.134182
             R2    0.821808  0.778838
Correlation (R)    0.910522  0.883827

TOP 10 COMBINATIONS
   metric  n_neighbors  p  weights  mean_test_score   CV_rmse
minkowski            3  1 distance      -253.261086 15.914179
manhattan            4  2 distance      -255.239774 15.976225
manhattan            4  1 distance      -255.785925 15.993309
minkowski            4  1 distance      -256.070881 16.002215
manhattan            3  1 distance      -256.432087 16.013497
manhattan            3  2 distance      -257.944586 16.060653
manha

### 6.2 SVR - Pipeline & Tuning

In [9]:
# SVR PIPELINE CREATION
svr_pipeline = Pipeline([('preprocessor', preprocessor), ('model', SVR(kernel='rbf'))])

svr_grid = {
    'model__C':        [512, 1024, 2048, 4096],
    'model__epsilon': [0.01, 0.1, 0.5, 1, 3],
    'model__gamma': ['scale', 0.001, 0.01, 0.1, 1]
}

gs_svr = run_grid_search(svr_pipeline, svr_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'SVR')


GridSearchCV will test 100 combinations for SVR...

BEST HYPERPARAMETERS — SVR
  C: 1024
  epsilon: 3
  gamma: scale
Best CV RMSE: 14.7073

VALIDATION SET PERFORMANCE
RMSE: 14.1309
MAE: 9.7122
R2: 0.8597
Correlation: 0.9300

TEST SET PERFORMANCE
RMSE: 13.1096
MAE: 9.2133
R2: 0.8531
Correlation: 0.9247

RESULTS SUMMARY
          Metric  Validation      Test
           RMSE   14.130917 13.109632
            MAE    9.712172  9.213344
             R2    0.859699  0.853058
Correlation (R)    0.930036  0.924683

TOP 10 COMBINATIONS
   C  epsilon gamma  mean_test_score   CV_rmse
1024     3.00 scale      -216.303556 14.707262
 512     3.00   0.1      -218.198433 14.771541
 512     3.00 scale      -218.539788 14.783091
1024     1.00 scale      -218.668072 14.787430
1024     0.01 scale      -220.158410 14.837736
 512     0.10   0.1      -220.371989 14.844931
 512     1.00 scale      -221.576607 14.885450
 512     0.01 scale      -222.117050 14.903592
 512     0.10 scale      -222.208797 14.9066

### 6.3 NuSVR - Pipeline & Tuning

In [10]:
# NuSVR PIPELINE CREATION
nusvr_pipeline = Pipeline([('preprocessor', preprocessor), ('model', NuSVR(kernel='rbf'))])

nusvr_grid = {
    'model__C':   [512, 1024, 2048, 4096],
    'model__nu': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    'model__gamma': ['scale', 0.001, 0.01, 0.1, 1]
}

gs_nusvr = run_grid_search(nusvr_pipeline, nusvr_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'NuSVR')


GridSearchCV will test 180 combinations for NuSVR...

BEST HYPERPARAMETERS — NuSVR
  C: 1024
  gamma: scale
  nu: 0.4
Best CV RMSE: 14.6345

VALIDATION SET PERFORMANCE
RMSE: 14.3086
MAE: 9.7774
R2: 0.8561
Correlation: 0.9278

TEST SET PERFORMANCE
RMSE: 13.2144
MAE: 9.1518
R2: 0.8507
Correlation: 0.9230

RESULTS SUMMARY
          Metric  Validation      Test
           RMSE   14.308609 13.214399
            MAE    9.777434  9.151794
             R2    0.856148  0.850700
Correlation (R)    0.927778  0.923045

TOP 10 COMBINATIONS
   C gamma  nu  mean_test_score   CV_rmse
1024 scale 0.4      -214.169390 14.634527
1024 scale 0.3      -217.080327 14.733646
 512 scale 0.8      -217.283425 14.740537
 512   0.1 0.2      -217.652750 14.753059
 512 scale 0.3      -218.549082 14.783406
 512 scale 0.5      -218.732428 14.789605
2048 scale 0.3      -219.968980 14.831351
1024 scale 0.5      -220.516483 14.849797
 512 scale 0.6      -220.571880 14.851663
 512   0.1 0.4      -220.651982 14.854359


## 7. Feature Analysis

**Preprocessing Pipeline:**
```python
preprocessor.fit(X_train, y_train)
X_transformed = preprocessor.transform(X_train)
```
Summary statistics of preprocessed features

In [11]:
target_col = 'cs_28d'
X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(df_recoded_50, target_col)

X = df_recoded_50.drop(columns=[target_col])
numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)
preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns)
preprocessor.fit(X_train, y_train)

# get column names
ohe_cols = preprocessor.named_steps['preprocessor'].named_transformers_['ohe'].get_feature_names_out(one_hot_columns).tolist()
all_cols = numerical_cols + ohe_cols + k_fold_columns

X_transformed = pd.DataFrame(
    preprocessor.transform(X_train),
    columns=all_cols
)

summary = pd.DataFrame({
    'mean':     X_transformed.mean().round(2),
    'median':   X_transformed.median().round(2),
    'std':      X_transformed.std().round(2),
    'min':      X_transformed.min().round(2),
    'max':      X_transformed.max().round(2),
    'cv (%)':   (X_transformed.std() / X_transformed.mean() * 100).round(1),
    'skewness': X_transformed.skew().round(2),
    'missing':  X_transformed.isna().sum()
})

print(summary.to_string())

                                  mean  median   std   min    max        cv (%)  skewness  missing
cement                           -0.00   -0.55  1.00 -0.86   4.21 -6.239880e+04      1.38        0
silica_fume                      -0.01    0.06  1.03 -9.15   4.27 -7.966200e+03     -3.37        0
fly_ash                          -0.01    0.03  1.00 -4.19   5.12 -1.038300e+04      0.80        0
limestone_powder                 -0.01    0.71  1.01 -3.64   1.76 -1.200120e+04     -1.22        0
quartz_powder                     0.00    0.03  1.00 -3.01   5.27  1.803801e+17      0.48        0
glass_powder                     -0.00    0.11  1.00 -1.91   4.78 -1.266856e+18      0.43        0
rice_husk_ash                     0.00   -0.42  1.00 -0.42   7.45  1.167317e+18      2.81        0
metakaolin                       -0.00   -0.23  1.00 -0.23  15.32 -1.702338e+18      6.32        0
ggbfs                            -0.00   -0.53  1.00 -0.53   4.04 -1.878441e+18      1.72        0
slag      

## 8. Data Info & Missing Values

In [12]:
print(df_recoded_50.shape)
print(df_recoded_50.isnull().mean().sort_values(ascending=False).head(20))

(2073, 34)
sand_max_size       0.133623
curing_temp         0.076218
sp_amount           0.003859
cement              0.000000
cement_type         0.000000
silica_fume         0.000000
quartz_powder       0.000000
fly_ash             0.000000
fly_ash_type        0.000000
limestone_powder    0.000000
ggbfs               0.000000
slag                0.000000
slag_type           0.000000
nano_caco3          0.000000
nano_al2o3          0.000000
glass_powder        0.000000
rice_husk_ash       0.000000
metakaolin          0.000000
filler              0.000000
nano_sio2           0.000000
dtype: float64


## 9. Error Analysis by Category

Evaluates predictions from all 3 best models (KNN, SVR, NuSVR)

In [13]:
for gs, model_name in zip([gs_knn, gs_svr, gs_nusvr], ['KNN', 'SVR', 'NuSVR']):
    X_test_df = X_test.copy()
    X_test_df['y_true'] = y_test.values
    X_test_df['y_pred'] = gs.best_estimator_.predict(X_test)
    X_test_df['abs_error'] = abs(X_test_df['y_true'] - X_test_df['y_pred'])
    
    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print('='*50)
    
    print("\n-- Cement Type --")
    print(X_test_df.groupby('cement_type')['abs_error'].mean().sort_values(ascending=False))
    
    print("\n-- Fiber vs No Fiber --")
    X_test_df['has_fiber'] = (X_test['fiber1_amount'] > 0).map({True: 'fiber', False: 'no_fiber'})
    print(X_test_df.groupby('has_fiber')['abs_error'].agg(['mean', 'count']))
    
    print("\n-- Curing Method --")
    print(X_test_df.groupby('curing_method')['abs_error'].mean().sort_values(ascending=False))


Model: KNN

-- Cement Type --
cement_type
Unknown         28.184588
white_cement    28.182270
OPC_53          16.006268
HS_cement       15.598566
OPC_III         14.662673
OPC_I           11.828368
OPC_42.5         9.933025
OPC_52.5         8.871211
CEM_II           6.178423
Name: abs_error, dtype: float64

-- Fiber vs No Fiber --
                mean  count
has_fiber                  
fiber      10.869999    212
no_fiber   11.699907     99

-- Curing Method --
curing_method
Autoclave Curing    23.002487
Heat Curing         13.666385
Steam Curing        12.118073
Water Curing        10.638452
Standard Curing      9.964475
Hot Water Curing     9.391054
Name: abs_error, dtype: float64

Model: SVR

-- Cement Type --
cement_type
OPC_53          12.478903
Unknown         11.266950
OPC_I           10.995598
HS_cement       10.588600
OPC_42.5         8.673711
OPC_52.5         7.627959
OPC_III          6.032991
CEM_II           3.532789
white_cement     3.025423
Name: abs_error, dtype: float6

In [14]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_val_score

y = df_recoded_50[target_col]

model_log = TransformedTargetRegressor(
    regressor=gs_svr.best_estimator_,  # your fitted/pipeline SVR or NuSVR
    func=np.log1p,
    inverse_func=np.expm1
)

scores = cross_val_score(model_log, X, y, scoring='neg_root_mean_squared_error', cv=5)
print("CV RMSE with log target:", -scores.mean())

#MAKES PERFORMANCE WORST

CV RMSE with log target: 36.27328705190347


In [20]:
best_params_knn = gs_knn.best_params_
best_params_svr = gs_svr.best_params_
best_params_nusvr = gs_nusvr.best_params_

from pathlib import Path

path = Path("results.json")

with open(path) as f:
    data = json.load(f)

data["best_params"] = data.get("best_params", {})
data["best_params"]["recoded_50"] = {
    "knn": best_params_knn,
    "svr": best_params_svr,
    "nusvr": best_params_nusvr,
}

with open(path, "w") as f:
    json.dump(data, f, indent=2)

print(data["best_params"])

{'best_params_publications_included': {'knn': {'model__metric': 'manhattan', 'model__n_neighbors': 8, 'model__p': 1, 'model__weights': 'distance'}, 'svr': {'model__C': 1024, 'model__epsilon': 3}, 'nusvr': {'model__C': 512, 'model__nu': 0.4}}, 'recoded_50': {'knn': {'model__metric': 'minkowski', 'model__n_neighbors': 3, 'model__p': 1, 'model__weights': 'distance'}, 'svr': {'model__C': 1024, 'model__epsilon': 3, 'model__gamma': 'scale'}, 'nusvr': {'model__C': 1024, 'model__gamma': 'scale', 'model__nu': 0.4}}}
